<a href="https://colab.research.google.com/github/lamboant01/maie383/blob/main/lab01_introduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 - introduction

Follow these examples with the demonstrator during the first 15 minutes. You can also rehearse them before the lab within the 30–45-minute preparation time.

1. Download and extract **lab01_student_kit.zip**. This notebook is **lab01_introduction.ipynb**.
2. In [Google Colab](https://colab.research.google.com/), upload this notebook and save your own copy.
3. Run the supplied setup cell; choose the ZIP when asked. Use a **CPU** runtime.
4. Run the small examples in order. Change one value, predict the result, and rerun.
5. Leave **PRACTICE** selected in the final form. Choose a branch and inspect its recording overview. No issued token or section code is needed for this rehearsal.

Finish by saving and reopening your notebook. Check that your changed code and output are still present.

This preparation has **no submission or marks**. It is attached to Lab 1, with no additional scheduled lab. Use lab01_student.ipynb for the assessed session; the same support ZIP serves both notebooks.


In [ ]:
from pathlib import Path
import sys, json, zipfile
_roots = [Path.cwd(), Path.cwd()/'miae383_lab01']
ROOT = next((p for p in _roots if (p/'src/miae383/lab01_data.py').is_file()
             and (p/'data/lab01/manifest.json').is_file()), None)
if ROOT is None:
    candidates = sorted(Path.cwd().glob('*lab01*kit*.zip'))
    if not candidates:
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError('Extract lab01_student_kit.zip, open this notebook from that folder, and rerun.')
        print('Choose lab01_student_kit.zip from your downloads.')
        uploaded = files.upload()
        candidates = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
    if len(candidates) != 1:
        raise RuntimeError('Choose exactly one Lab 1 support ZIP. Keep only the current copy in this runtime.')
    ROOT = (Path.cwd()/'miae383_lab01').resolve()
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(candidates[0]) as archive:
        if 'LAB01_KIT.json' not in archive.namelist():
            raise RuntimeError('This is not the Lab 1 student kit. Select lab01_student_kit.zip.')
        marker = json.loads(archive.read('LAB01_KIT.json'))
        if marker['version'] != 'lab01-recordings-2026-09-09':
            raise RuntimeError('This kit belongs to an older lab. Download the current one from Moodle.')
        for member in archive.infolist():
            target = (ROOT/member.filename).resolve()
            if not target.is_relative_to(ROOT):
                raise RuntimeError('The ZIP contains an invalid path. Request a fresh copy.')
        archive.extractall(ROOT)
sys.path.insert(0,str(ROOT/'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from miae383.identity import ViewContext, is_preview_identity, check_submission_identity
from miae383.lab01_data import assigned_recordings, overview
from miae383.lab01_support import check, check_all, export_available_work
print('Lab 1 setup complete. CPU runtime is sufficient.')
print('Continue with the next cell.')


## Opening tutorial (about 15 minutes)

Follow the demonstrator, then try the examples.

Use **Shift+Enter** to run the selected cell. A number in brackets shows that a code cell has run. After editing a function, run its cell again before using it.

In [ ]:
pressure_bar = [10., 12., 14.]
print('First reading:', pressure_bar[0], 'bar')
print('Number of readings:', len(pressure_bar))
print('Pressure range:', max(pressure_bar)-min(pressure_bar), 'bar')


Change the last reading from 14 to 18 and rerun: the range becomes 8 bar. Undo the change and rerun.

In [ ]:
def pressure_span(values):
    return max(values)-min(values)

print(pressure_span([10., 12., 14.]))
assert pressure_span([10., 12., 14.]) == 4.
print('Known-answer check passed.')


The next cell asks for position 3, but this list has positions 0, 1 and 2. Read the error, change 3 to 2 and rerun.

In [ ]:
position = 3
try:
    print('Requested reading:', pressure_bar[position])
except IndexError as error:
    print('IndexError:', error)
    print('Three items use positions 0, 1, 2. Change position to 2, then rerun this cell.')


`return` passes a result back to the caller. Edit the five marked function bodies and your response boxes; the other cells are supplied.

Keep a saved notebook for each student. Rename it `Lab1_<your-token-prefix>.ipynb` and run it with your own token. Both partners must explain their results. Download the notebook and results.json before leaving.

## Load your assigned recordings

Choose one case with your partner. The core tasks and marks are the same.

**Mechanical:** **Milling machine:** compare processed table vibration (vib_table) with spindle current (smcAC). Investigate cutting recordings and a peak near the documented cutter frequency. Segments are 8 s at 250 Hz.

**Aerospace:** **Aircraft:** compare pitch and roll during climb, cruise and approach. Measurements are in degrees, sampled at 8 Hz over 250 s segments.

**Industrial:** **Hydraulic equipment:** compare PS1/PS2 pressure cycles across component conditions. Investigate which records an outlier rule removes. Measurements are in bar, sampled at 100 Hz over 60 s cycles.

No codes yet? Leave **DEMO** in both fields to practise. Before submitting, enter your issued four-letter token and section code, untick **PRACTICE**, then restart and run all cells. Your assigned data and answers may change.

In [ ]:
#@title Your Lab 1 details - change these fields, then run
PRACTICE = True #@param {type:"boolean"}
# No token yet? Use DEMO to explore the lab.
STUDENT_TOKEN = "DEMO" #@param {type:"string"}
SECTION_ID = "AI" #@param ["AI", "BI", "CI", "DI", "FI"]
BRANCH = "mechanical" #@param ["mechanical", "aerospace", "industrial"]
# No section code yet? Use DEMO to explore the lab.
RELEASE_CODE = "DEMO" #@param {type:"string"}
PAIR_ID = "" #@param {type:"string"}

# Before submitting: replace BOTH defaults with your issued four-letter codes; untick PRACTICE.
# Then restart and run all cells; update your answers for your assigned data.
_token = 'DEMO' if PRACTICE else STUDENT_TOKEN.strip()
_release = 'DEMO' if PRACTICE else RELEASE_CODE.strip()
try:
    context = ViewContext(_token,1,SECTION_ID,BRANCH,_release)
except (TypeError,ValueError) as error:
    raise ValueError('Enter your four-letter student token and section code, or DEMO to practise. '
                     +str(error)) from None
view = assigned_recordings(context, ROOT/'data')
SEED = view['metadata']['model_seed']
_completed_stages = {}
results = {'view_id':context.public_manifest()['view_id']}
feature_table = None
print('PREVIEW: replace both DEMO values, untick PRACTICE, then restart and run all cells before submitting.'
      if PRACTICE or is_preview_identity(context.student_token, context.release_code)
      else 'ASSIGNED VIEW - save these details in your notebook')
display(overview(view))
